In [2]:
import logging
import time
import pandas as pd
import requests

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler("api_project.log", encoding="utf-8"),logging.StreamHandler()])
logger = logging.getLogger(__name__)

In [4]:
BASE_URL = "https://open.er-api.com/v6/latest"
SELECTED_CURRENCIES = ["AZN", "EUR", "GBP", "TRY", "RUB"]

In [6]:
def send_request(url, retries=3, timeout=10):
    for attempt in range(1, retries + 1):
        try:
            logger.info("Sending request (attempt %s/%s): %s", attempt, retries, url)
            response = requests.get(url, timeout=timeout)
            status_code = response.status_code

            # 1. Success (200 OK)
            if status_code == 200:
                logger.info("Successful response, status code: %s", status_code)
                return response.json()

            # 2. Client-side errors (400-499). Retrying is meaningless.
            if 400 <= status_code < 500:
                logger.error("Client error (4xx), status code: %s | URL: %s", status_code, url)
                return None

            # 3. Server-side errors (500 and above). Will retry after waiting.
            if status_code >= 500:
                logger.warning("Server error (5xx), status code: %s", status_code)

        # Handling specific exceptions
        except requests.exceptions.Timeout:
            logger.warning("Timeout: server did not respond within %s seconds", timeout)
        except requests.exceptions.ConnectionError:
            logger.warning("Connection error: failed to connect to the server")
        except ValueError:
            logger.error("Response is not in valid JSON format")
            return None
        except requests.exceptions.RequestException as error:
            logger.error("Unexpected request error: %s", error)
            return None

        # Increasing wait time after each failed attempt (Exponential Backoff effect)
        time.sleep(2 * attempt)

    logger.error("All attempts failed: %s", url)
    return None


In [7]:

def get_exchange_rates_table(base_currency="USD"):
    data = send_request(f"{BASE_URL}/{base_currency}")

    if data is None:
        return None

    # Even if HTTP status is 200, the API might return an internal error
    if data.get("result") != "success":
        logger.error("API returned an error: %s", data.get("error-type", "unknown error"))
        return None

    rates = data["rates"]
    # Filter only our selected currencies from the dictionary
    selected_rates = {curr: rates[curr] for curr in SELECTED_CURRENCIES if curr in rates}

    # Creating a Pandas DataFrame
    df = pd.DataFrame(list(selected_rates.items()), columns=["currency", "rate"])
    df.insert(0, "base_currency", base_currency)
    df.insert(2, "last_updated", data.get("time_last_update_utc"))
    return df

In [8]:

def run_error_tests():
    """Sends requests to intentional error URLs to test error handling."""
    logger.info("--- Error test starting ---")
    send_request("https://httpbin.org/status/404")
    send_request("https://httpbin.org/status/500", retries=2)
    logger.info("--- Error test finished ---")



In [9]:

def main():
    # Fetching exchange rates based on USD
    df = get_exchange_rates_table("USD")

    if df is None:
        logger.error("No data received, program terminated")
        return

    # Displaying results in the terminal
    print("\nExchange Rates (for 1 USD):")
    print(df.to_string(index=False))

    # Simple analysis: calculating how much of other currencies 1 AZN can buy
    azn_rate = df.loc[df["currency"] == "AZN", "rate"].iloc[0]
    df["1_AZN_to_foreign"] = (df["rate"] / azn_rate).round(4)

    # Saving results to a CSV file (utf-8-sig ensures proper character encoding in Excel)
    output_filename = "exchange_rates.csv"
    df.to_csv(output_filename, index=False, encoding="utf-8-sig")
    logger.info("Results saved to %s (%s rows)", output_filename, len(df))

    # Uncomment the line below if you want to run error tests:
    # run_error_tests()


if __name__ == "__main__":
    main()

2026-11-09 15:39:23,650 | INFO | Sending request (attempt 1/3): https://open.er-api.com/v6/latest/USD
2026-11-09 15:39:24,828 | INFO | Successful response, status code: 200
2026-11-09 15:39:24,864 | INFO | Results saved to exchange_rates.csv (5 rows)



Exchange Rates (for 1 USD):
base_currency currency                    last_updated      rate
          USD      AZN Mon, 09 Nov 2026 00:02:31 +0000  1.700549
          USD      EUR Mon, 09 Nov 2026 00:02:31 +0000  0.896919
          USD      GBP Mon, 09 Nov 2026 00:02:31 +0000  0.748366
          USD      TRY Mon, 09 Nov 2026 00:02:31 +0000 50.192892
          USD      RUB Mon, 09 Nov 2026 00:02:31 +0000 81.928764
